# FSD50K audio for sound-search-demo

Makes `fsd50k_demo.zip`: the first 10 s (the embedded window) of each of the 2,399 FSD50K clips, as mp3 (~200 MB).

1. Reuses the FSD50K eval zip parts in Google Drive (`MyDrive/audio-vocab-fsd50k/`) from the earlier run if they are complete; otherwise downloads them from Zenodo (6.3 GB, resumable, ~10 min).
2. Joins the two-part zip, extracts only the 2,399 clips, converts them, and downloads the zip.

**Runtime -> Run all** (CPU is fine). Allow Google Drive access when asked.

In [ ]:
from google.colab import drive, files
import os, subprocess, time, zipfile, requests
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/audio-vocab-fsd50k'
LOCAL = '/content/fsd_local'; os.makedirs(LOCAL, exist_ok=True)
OUT = '/content/fsd50k_demo'; os.makedirs(OUT, exist_ok=True)
IDS = ["196669", "108530", "117117", "353105", "330835", "334980", "275798", "188038", "195346", "403217", "1926", "339159", "417223", "196645", "131267", "264523", "398058", "121565", "97333", "328053", "67305", "178889", "320273", "1942", "340049", "328660", "267504", "407634", "177252", "430816", "417076", "117181", "418782", "119633", "209537", "340887", "148928", "257438", "346898", "326964", "182854", "332129", "93031", "242498", "153054", "98063", "405572", "271602", "320059", "406613", "235595", "118337", "347562", "403213", "125299", "181936", "37242", "414415", "186698", "121346", "155092", "118817", "223366", "93847", "96078", "211667", "427687", "243479", "223610", "248235", "323165", "391917", "411451", "85067", "399198", "75105", "414358", "338322", "191901", "391516", "321230", "60918", "424791", "418396", "131264", "117303", "205599", "244985", "347854", "135717", "395308", "186475", "410241", "103033", "393764", "81875", "45907", "122802", "376509", "320234", "94774", "207451", "118999", "369265", "386759", "23533", "24220", "84924", "334983", "434155", "380810", "178615", "366818", "173946", "162803", "62459", "266626", "340919", "173440", "170546", "56258", "377084", "83509", "192647", "93030", "232557", "206144", "344591", "66583", "241278", "395472", "240254", "410866", "383782", "222648", "422145", "251898", "393513", "104305", "178751", "366382", "141269", "381943", "365632", "261013", "170284", "94213", "268143", "366407", "78957", "26184", "110107", "424739", "71731", "394656", "253605", "391974", "132902", "156640", "274471", "389751", "408117", "244457", "201212", "433599", "432106", "144166", "170464", "81557", "334992", "179339", "119958", "377572", "326377", "350769", "323794", "157580", "353793", "253754", "212427", "395820", "119634", "99185", "114725", "125780", "83741", "251907", "64617", "90790", "17138", "403214", "408562", "213986", "83465", "117121", "328761", "235519", "327688", "326102", "164805", "403675", "214356", "61577", "425140", "340867", "275788", "266001", "334618", "390710", "431599", "430011", "56396", "345138", "98944", "393498", "75188", "427182", "249715", "119831", "114407", "427414", "14577", "413155", "179696", "398885", "416960", "403230", "390250", "165618", "393900", "379088", "351479", "76732", "169678", "430814", "61503", "106245", "345737", "330848", "360450", "108017", "240280", "119088", "118521", "95517", "125173", "347537", "154753", "60909", "156639", "328130", "130585", "173989", "415354", "274103", "415763", "65923", "406241", "331646", "407605", "331907", "185310", "206167", "274121", "13594", "325449", "433735", "340898", "26762", "28746", "242138", "13797", "48241", "393179", "337723", "181344", "367437", "14829", "328394", "408742", "159429", "338147", "324485", "62335", "387904", "172639", "168730", "26769", "74485", "104293", "140055", "135191", "390244", "396311", "317078", "410384", "397952", "156064", "419421", "426571", "187687", "348310", "23229", "429401", "397673", "2525", "123306", "340901", "235537", "90421", "402452", "124720", "207485", "152716", "273929", "21189", "96503", "72560", "93314", "84387", "257204", "14833", "407668", "49777", "37240", "25293", "269260", "388357", "25301", "23707", "257620", "49774", "330834", "97605", "365633", "334978", "111398", "288930", "19094", "382991", "254819", "325947", "171447", "81117", "361148", "325942", "36061", "108750", "59462", "393379", "331024", "69342", "191118", "59619", "342654", "93800", "273450", "344576", "171324", "336370", "377083", "84353", "410172", "424502", "70052", "62468", "257436", "14575", "180338", "59454", "50178", "91113", "350619", "346108", "177619", "37245", "348190", "370994", "400888", "184372", "399799", "58548", "52053", "17063", "54187", "328813", "240272", "383143", "339141", "341568", "215174", "156641", "261020", "179349", "39192", "421712", "64603", "325453", "352418", "346168", "394891", "346146", "85475", "327389", "328712", "86853", "330025", "348326", "141528", "77148", "231963", "210262", "267501", "253598", "267427", "204717", "340465", "104813", "245596", "170287", "387905", "122814", "131237", "119113", "345803", "119026", "93309", "68999", "153143", "383783", "425153", "125940", "325185", "106559", "16774", "198612", "90742", "221548", "243467", "261431", "320658", "172738", "78369", "62458", "117973", "27072", "424872", "395830", "327002", "118799", "380040", "9098", "377065", "263048", "174509", "81853", "346473", "108387", "33847", "168030", "320070", "133473", "401539", "99634", "147325", "402993", "376638", "245434", "156635", "264842", "73224", "407764", "249718", "189567", "37618", "317665", "50152", "110582", "118901", "197122", "21312", "16783", "377582", "392259", "26217", "338168", "327396", "82610", "191520", "234044", "51393", "407587", "210734", "58369", "209989", "155942", "154840", "367823", "399652", "414292", "321456", "1928", "353185", "409246", "365578", "341867", "134909", "191731", "425776", "163383", "110106", "170282", "205079", "268132", "178413", "195178", "80601", "58382", "101746", "348322", "238016", "352413", "205091", "400590", "348331", "366028", "257375", "371602", "72240", "431610", "366318", "416538", "428668", "395626", "88447", "189773", "344220", "172753", "341550", "345853", "395296", "321465", "75404", "47874", "277005", "256681", "96027", "416226", "186770", "215572", "135429", "335020", "99694", "327057", "326960", "216499", "253881", "427772", "180377", "69385", "396985", "127994", "140301", "405431", "162045", "102563", "369702", "89287", "431873", "423627", "396982", "89290", "190647", "219532", "79256", "156098", "326296", "389485", "81556", "349893", "264046", "423092", "386734", "365636", "201914", "149984", "263078", "242006", "188817", "321200", "341908", "110537", "344574", "394181", "427113", "73144", "234237", "246037", "213260", "211191", "191411", "204606", "221398", "408735", "400183", "232731", "416537", "152595", "389997", "399830", "152718", "386717", "60907", "240263", "249717", "170876", "255855", "371339", "365764", "178417", "344438", "425663", "215937", "391300", "340861", "396270", "399800", "112064", "350714", "25713", "171254", "51054", "367744", "182812", "365495", "92023", "93052", "388791", "186096", "347557", "328908", "25822", "263992", "340869", "382572", "395684", "62025", "186356", "155467", "222899", "51059", "276535", "391277", "196693", "391748", "88490", "1013", "243022", "154402", "131202", "405262", "369293", "17151", "134688", "404024", "237766", "395124", "202231", "73419", "216659", "106555", "244282", "428075", "26296", "340872", "363143", "132900", "38688", "368006", "115611", "83742", "266394", "29621", "402316", "69307", "128648", "178612", "106018", "64602", "30823", "366692", "91114", "213943", "85683", "179675", "85238", "215943", "97756", "196142", "33844", "127153", "86082", "44369", "233067", "185050", "262214", "405212", "2539", "107125", "412769", "103073", "340910", "103076", "71220", "222752", "211528", "178313", "326122", "77026", "369701", "118807", "259925", "388857", "344335", "327391", "405327", "233062", "96936", "427365", "400805", "431533", "397020", "325532", "135873", "407616", "90416", "179672", "174472", "403908", "113648", "345836", "223273", "118565", "57801", "403562", "94227", "326347", "366763", "223277", "189971", "86044", "258263", "345738", "327049", "56533", "157433", "431320", "407509", "411438", "427700", "390313", "56380", "60987", "189766", "186147", "95087", "166433", "39319", "134742", "135646", "16451", "132602", "131266", "43865", "257864", "85681", "211190", "365634", "277003", "104275", "59455", "62327", "399197", "197223", "368014", "408601", "434204", "47050", "235507", "245948", "400393", "45603", "189764", "378689", "39436", "388360", "416196", "62594", "231957", "391842", "199893", "360620", "26287", "197989", "317330", "349846", "352613", "18149", "344596", "421608", "89210", "255565", "389613", "50389", "429397", "135000", "17152", "213987", "127995", "118796", "409102", "325459", "271947", "146771", "393508", "370938", "369419", "428103", "186374", "173117", "380753", "213911", "77926", "389761", "18449", "235706", "117979", "351285", "320571", "367438", "233256", "413550", "336772", "147692", "125717", "257865", "419223", "180334", "326373", "397920", "85454", "34207", "408421", "125169", "417736", "332211", "31571", "35220", "315840", "26712", "347150", "421615", "345023", "141188", "361029", "78382", "347855", "424596", "62476", "167699", "399828", "119024", "170726", "84943", "360727", "45915", "130777", "406088", "31542", "316797", "380482", "69299", "259192", "429264", "11503", "57619", "406242", "212038", "429970", "135860", "62947", "222671", "395314", "258256", "159492", "180532", "181261", "38116", "244291", "54545", "122809", "25270", "124523", "85712", "207135", "173401", "269271", "159460", "380811", "338171", "417515", "425662", "379115", "316869", "280243", "98859", "254822", "152994", "390324", "366149", "145338", "250840", "54276", "93029", "105277", "185521", "166921", "329776", "185048", "329986", "82538", "361333", "213261", "408623", "390005", "367125", "156510", "334619", "398007", "257006", "335082", "88454", "118577", "186391", "425150", "413462", "347967", "339085", "272383", "76676", "383335", "117978", "415352", "396491", "352625", "352632", "205015", "132472", "424774", "164803", "365710", "257691", "218846", "367039", "414494", "208765", "399656", "328894", "35364", "330854", "58665", "337815", "221787", "82422", "365637", "56532", "208995", "209800", "317012", "408499", "411014", "220045", "416840", "420679", "368684", "338426", "365103", "278226", "96138", "78520", "383839", "250355", "35214", "45809", "130794", "161657", "127450", "15577", "210390", "405261", "201899", "321753", "386749", "33676", "251931", "330804", "144171", "355741", "108470", "422137", "264918", "272410", "366402", "415026", "161698", "333917", "57696", "52052", "191850", "121946", "136052", "217611", "405531", "72840", "8380", "423626", "187331", "391284", "349393", "155793", "334982", "70878", "417998", "53447", "214989", "153239", "238204", "78435", "162793", "35835", "410572", "406243", "330956", "34979", "154856", "158971", "386452", "175627", "269251", "96929", "72950", "423777", "219856", "239492", "205732", "217740", "164764", "392881", "30939", "123305", "214343", "392258", "390338", "16940", "194926", "331428", "57695", "416507", "187334", "353665", "233259", "325643", "7397", "154470", "366531", "424736", "79886", "180252", "340897", "88528", "403215", "416969", "1017", "403321", "213983", "425664", "167505", "383785", "257867", "188540", "218999", "365104", "135859", "51206", "345903", "265579", "240264", "369321", "362354", "58489", "131199", "249745", "64460", "217172", "350283", "386718", "342166", "223000", "277008", "266191", "338197", "180924", "22718", "324108", "76590", "366102", "207126", "271661", "118066", "31489", "238013", "58384", "262957", "120019", "120468", "221785", "117496", "90219", "408807", "364701", "17486", "393356", "253880", "59739", "178559", "411026", "88474", "77404", "340366", "418797", "352631", "320269", "410990", "168419", "325945", "376507", "124533", "194995", "64445", "31777", "393059", "187021", "210276", "250378", "380320", "369295", "343165", "131560", "24228", "131642", "127991", "234737", "159730", "85731", "136991", "187705", "185046", "219605", "190696", "324423", "49328", "371179", "235558", "42978", "179694", "210270", "246248", "333659", "96351", "269478", "240186", "401303", "232842", "355063", "62521", "336756", "72578", "257649", "331680", "351770", "261433", "366760", "219867", "423148", "171519", "424786", "118082", "62446", "315837", "271654", "33562", "400867", "47338", "380440", "240267", "142326", "173088", "329651", "101846", "167688", "102547", "145383", "62516", "171378", "266949", "426784", "71139", "33853", "185055", "157156", "325526", "428860", "269337", "154431", "16799", "57553", "108417", "419091", "103333", "185196", "42143", "104212", "22263", "203594", "324659", "362919", "382989", "198393", "372893", "159459", "179113", "257866", "60930", "171451", "408737", "326464", "76404", "335962", "320278", "58391", "137164", "250583", "154438", "115378", "411025", "327666", "351284", "152953", "124532", "255574", "265290", "383775", "408095", "128901", "102684", "246192", "185446", "52224", "325946", "382758", "163286", "366980", "406915", "394117", "34268", "344679", "11389", "220835", "216660", "34521", "366784", "426317", "382589", "410969", "31547", "389617", "326452", "60878", "31569", "82083", "29545", "399190", "42959", "164341", "122803", "68382", "181900", "344131", "102908", "119441", "25907", "170487", "402143", "69132", "130584", "403206", "327399", "47339", "382582", "196559", "346022", "17747", "272242", "386773", "377956", "331885", "365059", "201273", "124534", "390712", "85684", "31676", "382762", "175520", "157157", "73701", "62628", "214357", "326687", "196382", "27787", "117548", "218418", "351778", "206081", "31491", "363137", "142594", "382973", "21113", "152991", "248035", "381856", "15902", "86568", "69384", "198611", "431980", "407577", "120832", "351274", "59184", "169127", "65292", "324775", "425208", "234862", "323178", "253759", "408380", "259191", "137734", "275336", "328635", "378257", "369297", "379168", "328132", "21109", "277009", "366319", "327381", "162312", "350759", "337142", "161543", "366520", "210319", "389988", "347964", "332162", "192646", "337814", "332603", "179045", "369202", "118998", "187609", "321737", "123085", "17748", "376377", "362382", "396993", "383113", "347362", "15589", "394057", "210023", "419437", "23441", "57774", "35622", "271557", "408756", "424601", "365762", "115084", "94214", "365838", "388404", "72577", "138362", "37239", "208836", "337813", "128607", "232963", "143126", "156089", "321439", "414400", "54272", "352612", "41531", "133869", "330300", "58389", "73146", "52353", "328052", "119840", "395414", "34735", "186335", "410975", "117305", "402614", "83184", "337754", "420010", "82414", "365635", "194123", "198991", "164094", "98943", "339874", "424740", "412525", "393510", "59104", "73752", "86342", "232728", "28723", "209619", "210048", "401721", "17490", "416961", "257413", "328699", "66113", "427008", "44791", "86284", "50293", "411222", "408882", "404010", "240324", "235178", "360908", "275134", "244456", "330849", "418335", "85702", "205087", "396221", "351275", "256582", "77691", "403500", "131597", "323006", "250087", "28714", "59324", "124524", "70938", "124501", "110586", "22973", "233258", "56274", "400093", "334939", "196692", "121548", "70305", "425314", "376367", "408483", "208656", "166489", "29544", "191388", "336788", "335013", "25590", "104210", "194806", "254080", "116163", "154458", "363153", "342660", "417221", "331241", "96030", "332913", "390339", "73260", "218082", "414411", "341557", "25908", "253636", "54512", "330312", "351277", "240120", "415298", "74732", "327438", "106972", "69386", "23442", "423990", "118436", "154440", "396986", "418423", "345861", "53991", "381783", "376502", "19477", "365670", "405592", "209525", "394223", "388272", "121801", "28209", "162816", "365132", "389992", "50665", "217853", "85680", "364972", "232357", "326980", "267573", "368013", "347960", "50454", "366773", "379170", "76220", "396530", "156907", "418422", "389638", "403564", "232107", "190033", "390295", "217806", "59578", "203164", "246060", "171456", "166258", "34395", "331396", "349690", "218839", "390270", "402729", "367808", "334937", "22694", "140054", "53268", "62591", "245459", "348968", "397621", "218840", "256579", "188032", "257007", "326916", "327681", "349902", "288937", "232788", "328146", "394658", "325701", "59513", "110389", "117972", "389759", "348129", "365566", "258148", "252712", "389615", "272093", "337435", "178377", "377742", "403462", "383879", "339940", "347400", "197327", "17204", "76135", "33854", "191733", "408703", "271605", "118072", "86344", "244503", "321581", "104802", "418281", "390285", "327421", "390473", "191152", "50455", "41308", "275339", "277382", "54987", "154788", "31840", "231959", "149258", "422263", "144184", "342656", "220375", "369549", "388765", "412558", "104277", "400804", "379186", "233205", "79598", "223611", "426594", "280251", "217804", "323164", "209969", "402549", "425779", "197106", "243659", "257859", "49904", "387833", "415201", "320449", "398977", "327053", "376499", "66717", "178560", "114026", "144181", "323174", "90649", "28692", "411664", "210362", "410237", "319105", "57584", "235594", "178898", "189337", "146932", "70885", "117648", "105380", "333229", "390167", "362916", "48935", "369194", "320061", "172552", "51364", "347853", "257005", "91366", "114503", "188767", "390302", "395396", "17124", "238522", "235905", "201706", "62477", "98843", "9922", "328892", "161214", "185258", "365569", "197222", "211872", "403980", "100839", "345813", "191392", "401930", "408746", "339927", "2538", "26402", "185457", "19475", "100030", "250389", "262265", "101807", "152803", "268753", "119832", "242538", "164247", "405205", "206080", "19260", "235622", "146291", "327559", "380398", "140297", "119501", "120034", "405573", "326961", "161326", "118440", "58323", "266610", "319403", "158620", "414027", "107228", "118087", "377589", "258265", "378654", "87724", "196655", "335401", "245862", "391747", "244452", "352504", "119437", "35837", "201895", "17081", "119116", "256570", "131641", "382903", "90785", "221788", "264461", "58015", "82319", "338212", "408490", "414357", "345073", "256452", "170883", "113175", "58488", "217138", "337632", "128746", "342733", "251941", "178850", "94676", "140454", "405432", "352643", "403531", "385892", "34150", "330853", "339990", "327917", "410174", "83915", "137894", "390287", "25309", "176126", "337816", "31540", "379416", "321671", "84966", "264361", "352347", "148855", "381761", "219289", "88475", "165617", "73373", "26763", "190031", "156091", "147036", "339932", "257206", "118566", "105385", "64556", "323795", "234681", "396983", "380646", "317014", "345746", "171486", "326491", "52221", "209620", "179364", "197326", "323793", "138373", "109710", "257435", "81179", "167507", "407686", "58666", "391923", "234866", "396220", "15792", "221851", "262605", "209539", "391520", "396996", "99580", "272237", "370814", "235511", "163289", "250770", "330601", "403881", "241272", "90381", "348024", "365986", "400798", "34545", "185629", "192553", "325534", "339086", "328906", "59586", "22693", "153150", "17008", "377602", "401722", "382258", "156095", "59512", "243480", "365060", "22459", "383728", "327916", "31505", "403981", "267711", "60989", "412527", "153090", "137411", "37244", "43481", "258074", "361135", "338170", "25308", "345155", "348509", "180536", "393763", "204604", "340905", "117120", "327417", "345092", "378908", "180180", "17894", "87275", "429284", "115869", "409230", "383784", "120010", "366070", "405590", "204052", "37929", "120017", "197916", "407568", "426954", "65997", "362503", "189669", "275103", "268760", "18442", "332066", "197957", "249490", "260720", "276469", "383777", "338695", "194460", "424870", "421025", "367078", "365102", "201722", "270655", "369598", "135903", "268795", "248036", "390342", "78372", "396757", "108023", "404399", "431396", "323627", "334938", "42206", "235554", "270346", "41325", "408263", "396483", "233473", "428400", "59453", "320167", "404398", "407745", "246190", "376753", "51058", "378892", "325462", "369707", "182307", "230362", "43806", "360371", "218048", "346610", "431651", "213982", "101468", "25302", "110611", "1043", "330400", "85771", "235962", "404142", "434156", "119073", "181171", "420887", "197130", "422262", "364707", "321203", "32320", "119556", "153389", "235363", "338686", "114305", "338169", "207782", "396464", "425680", "260626", "58368", "119864", "419550", "276577", "217609", "39855", "350755", "199929", "209970", "120501", "408822", "53471", "18271", "70907", "20351", "406995", "95086", "353111", "95085", "147321", "336798", "220717", "110536", "35623", "152312", "378897", "250382", "389518", "202420", "272174", "344606", "179998", "182785", "410964", "110610", "413715", "378702", "15555", "49903", "96080", "60877", "260933", "404387", "9197", "418165", "231495", "186439", "396168", "135848", "201936", "407502", "147322", "390711", "413691", "331336", "236007", "334690", "101732", "124502", "58202", "127992", "343168", "349096", "377736", "56531", "127993", "407561", "328541", "320451", "404684", "408447", "144137", "153578", "18621", "73497", "420684", "192178", "415221", "60933", "407555", "349922", "420821", "59579", "317533", "251279", "189765", "30303", "414042", "43588", "148854", "198846", "328751", "26297", "206170", "377112", "143989", "274153", "62329", "408748", "401820", "29756", "397919", "380772", "235506", "22682", "380209", "107842", "334084", "157381", "327677", "399660", "333779", "238198", "144178", "378354", "32653", "186395", "210094", "420661", "59668", "115602", "389652", "222468", "364694", "59666", "39532", "109505", "1044", "381527", "347843", "404404", "149190", "35624", "1042", "170848", "188006", "144141", "425681", "18448", "108452", "48430", "408485", "37241", "275340", "393895", "49784", "210308", "344578", "191114", "21187", "19697", "366219", "415199", "119450", "141684", "204083", "62109", "395241", "167201", "185056", "388252", "16363", "427134", "352715", "402937", "268538", "153149", "119093", "366586", "418356", "60164", "353207", "398004", "137134", "218843", "76589", "93307", "179343", "393333", "152802", "201274", "119455", "412186", "337200", "320734", "185063", "394221", "171674", "233454", "1028", "70939", "50667", "32512", "406129", "328097", "184663", "260179", "334474", "402146", "352374", "367005", "154834", "365988", "353124", "52303", "378911", "265249", "218413", "58359", "118099", "168029", "256646", "382984", "365575", "387021", "135718", "59596", "188768", "185259", "244226", "408096", "78958", "81065", "380664", "93308", "27785", "329784", "275338", "366895", "380101", "41328", "59457", "39533", "90458", "423006", "22717", "25591", "160964", "393444", "119853", "59594", "407459", "131643", "272353", "32990", "39531", "347838", "371153", "389998", "328753", "390165", "393509", "318203", "383787", "377584", "345075", "361117", "167750", "207131", "135847", "58371", "146926", "328764", "147330", "390242", "360717", "57181", "147827", "19695", "366604", "96077", "119102", "135849", "32807", "95067", "70936", "399092", "194940", "32511", "417430", "161491", "143119", "391301", "121582", "408740", "383786", "223051", "55731", "366068", "87133", "370217", "398415", "339414", "146422", "73257", "39052", "61504", "405585", "57771", "261377", "257002", "150231", "343743", "328816", "359154", "409197", "257711", "115239", "216624", "189843", "412556", "142333", "50666", "389651", "223978", "332563", "166213", "189460", "274499", "396490", "320772", "234736", "36266", "409010", "174618", "141512", "408621", "348076", "16754", "318829", "94210", "69007", "318197", "149249", "320450", "323628", "368011", "415928", "194942", "402164", "411933", "319017", "341391", "17123", "168028", "118994", "427160", "402395", "17146", "31775", "118101", "198466", "417608", "106270", "244505", "259514", "75195", "174467", "200809", "345275", "121556", "389612", "110103"]
print(len(IDS), 'clips wanted')

BASE = 'https://zenodo.org/records/4060432/files/'
def size_of(name):
    r = requests.get(BASE + name + '?download=1', headers={'Range': 'bytes=0-0'}, timeout=60)
    return int(r.headers['content-range'].split('/')[-1])

def get(name, chunk=64 << 20):
    """Local copy of one zip part: from Drive if complete there, else a resumable download."""
    dst, size = f'{LOCAL}/{name}', size_of(name)
    if os.path.exists(dst) and os.path.getsize(dst) == size:
        return dst
    d = f'{WORK}/{name}'
    if os.path.exists(d) and os.path.getsize(d) == size:
        print('copying', name, 'from Drive ...'); subprocess.run(['cp', d, dst], check=True); return dst
    url, pos, t0 = BASE + name + '?download=1', (os.path.getsize(dst) if os.path.exists(dst) else 0), time.time()
    with open(dst, 'ab') as f:
        while pos < size:
            end = min(pos + chunk, size) - 1
            for attempt in range(20):
                try:
                    r = requests.get(url, headers={'Range': f'bytes={pos}-{end}'}, timeout=300)
                    if r.status_code == 206 and len(r.content) == end - pos + 1:
                        break
                except requests.RequestException:
                    pass
                time.sleep(5 * (attempt + 1))
            else:
                raise RuntimeError(f'download failed at {pos}')
            f.write(r.content); pos = end + 1
            print(f'\r{name}: {pos/1e9:.2f}/{size/1e9:.2f} GB ({pos/1e6/(time.time()-t0+1e-9):.0f} MB/s)', end='')
    print(); return dst

get('FSD50K.eval_audio.z01'); get('FSD50K.eval_audio.zip')
unsplit = f'{LOCAL}/unsplit.zip'
if not os.path.exists(unsplit):
    if subprocess.run(['which', 'zip'], capture_output=True).returncode != 0:
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'zip'], check=True)
    print('joining the two parts (a few minutes) ...')
    subprocess.run(['zip', '-q', '-s', '0', f'{LOCAL}/FSD50K.eval_audio.zip', '--out', unsplit], check=True)

with zipfile.ZipFile(unsplit) as z:
    names = {n.rsplit('/', 1)[-1][:-4]: n for n in z.namelist() if n.endswith('.wav')}
    missing = [c for c in IDS if c not in names]
    print('missing from the archive:', missing[:10])
    for k, c in enumerate(IDS):
        o = f'{OUT}/{c}.mp3'
        if c in names and not os.path.exists(o):
            w = f'{LOCAL}/{c}.wav'
            open(w, 'wb').write(z.read(names[c]))
            # first 10 s = the exact window that was embedded
            subprocess.run(['ffmpeg', '-nostdin', '-v', 'error', '-i', w, '-t', '10', '-ac', '1', '-b:a', '96k', o], check=True)
            os.remove(w)
        if k % 300 == 0: print(k, 'of', len(IDS))
print(len(os.listdir(OUT)), 'mp3s (expect 2399)')
subprocess.run(['zip', '-qrj', '/content/fsd50k_demo.zip', OUT], check=True)
print(round(os.path.getsize('/content/fsd50k_demo.zip') / 1e6), 'MB')
files.download('/content/fsd50k_demo.zip')
